In [1]:
# import packages
# ===============================
# import packages
# ===============================
import pandas as pd
import numpy as np
import requests
from datetime import datetime as dt, timezone, timedelta
from pathlib import Path

In [2]:
# ===============================
# system paths
# ===============================
script_dir = Path.cwd().parent
output_dir = script_dir / "data"

output_dir.mkdir(parents=True, exist_ok=True)

In [3]:
# ===============================
# 1 - Elexon generation data
# ===============================
# API request - generation data
url = 'https://data.elexon.co.uk/bmrs/api/v1/generation/outturn/summary'
today = dt.now(timezone.utc)
time_lag = timedelta(days=7)
start_date = pd.Timestamp(today - time_lag).round('D').strftime('%Y-%m-%d %H:%M:%S')
today = pd.Timestamp(today).strftime('%Y-%m-%d %H:%M:%S')

params = {
    'startTime' : start_date,
    'endTime' : today
}

response = requests.get(url, params = params)

if response.status_code == 200:
    print('Elexon generation API call successful!')
else:
    raise Exception(f"Elexon API failed. Status code: {response.status_code}")

gen = response.json()

gen = pd.json_normalize(
    gen,
    record_path=['data'],
    meta=['startTime', 'settlementPeriod']
)

# Convert data types - generation data
gen['settlementPeriod'] = pd.to_numeric(gen['settlementPeriod'])
gen['localTime'] = pd.to_datetime(gen['startTime'], utc=True).dt.tz_convert('Europe/London')
gen['startTime'] = gen['localTime'].dt.strftime('%Y-%m-%d')

gen = gen.pivot_table(
    index=['startTime', 'settlementPeriod'],
    columns='fuelType',
    values='generation'
).reset_index().rename_axis(None, axis=1)

Elexon generation API call successful!


In [4]:
# ===============================
# 2 - NESO demand data
# ===============================
# API request - NESO demand data
dmnd = pd.read_csv('https://api.neso.energy/dataset/7a12172a-939c-404c-b581-a6128b74f588/resource/177f6fa4-ae49-4182-81ea-0c6b35f26ca6/download/demanddataupdate.csv')

dmnd = pd.DataFrame(dmnd)

dmnd = dmnd.rename(
    columns={
        'SETTLEMENT_DATE': 'startTime',
        'SETTLEMENT_PERIOD': 'settlementPeriod'
    }
)

In [12]:
merged_data = pd.merge(dmnd, gen, on=['startTime', 'settlementPeriod'])


merged_data.columns

grid = merged_data[[
    'startTime',
    'settlementPeriod',
    'OIL',
    'CCGT',
    'OCGT',
    'COAL',
    'BIOMASS',
    'NUCLEAR',
    'WIND',
    'EMBEDDED_WIND_GENERATION',
    'EMBEDDED_SOLAR_GENERATION',
    'NPSHYD',
    'PS',
    'OTHER',
    'INTELEC',
    'INTFR',
    'INTIFA2',
    'INTNED',
    'INTIRL',
    'INTEW',
    'INTGRNL',
    'INTNEM',
    'INTNSL',
    'INTVKL'
]]

grid = grid.sort_values(by=['startTime', 'settlementPeriod'])

grid['order'] = range(1, len(grid) + 1)


In [14]:

print(grid.columns)

start_cols = ['order', 'startTime', 'settlementPeriod']

ordered_cols = start_cols + [c for c in grid.columns if c not in start_cols]

grid = grid[ordered_cols]

Index(['startTime', 'settlementPeriod', 'OIL', 'CCGT', 'OCGT', 'COAL',
       'BIOMASS', 'NUCLEAR', 'WIND', 'EMBEDDED_WIND_GENERATION',
       'EMBEDDED_SOLAR_GENERATION', 'NPSHYD', 'PS', 'OTHER', 'INTELEC',
       'INTFR', 'INTIFA2', 'INTNED', 'INTIRL', 'INTEW', 'INTGRNL', 'INTNEM',
       'INTNSL', 'INTVKL', 'order'],
      dtype='str')
